In [ ]:
# GEO Normal / Disease gene expression analysis

# Gene IDとGSE IDのみ変更すれば自動判定を試行できます。
GENE_ID = 327                        # NCBI Gene ID
GSE_ID = "GSE5281"                  # NCBI GEO Series
TECHNOLOGY = "auto"                 # auto / microarray / rna_seq

# Normal / Diseaseの2群をGSMの疾患・診断メタデータから自動判定
# 病名や疾患名の入力は不要。判定できないGSEは監査CSVで確認。
GROUP_MODE = "auto"                 # auto / csv
# GROUP_FIELD_OVERRIDE = ""          # 自動抽出する属性列を手動指定したい場合のみ
GROUP_FIELD_OVERRIDE = None
# POOL_MULTIPLE_DISEASE_GROUPS = True # 異なる疾患ラベルをすべてDiseaseへまとめる場合のみ
POOL_MULTIPLE_DISEASE_GROUPS = False
# GROUP_MODE = "csv"                 # 自動判定不可のときのみ有効化
GROUP_OVERRIDE_CSV = None           # gsm_id,group（Normal / Disease / Exclude）

# 任意の組織・採取条件フィルター（初期設定は無効）
COMMON_FILTERS = {}
GROUP_FILTERS = {}
INCLUDE_GSM_IDS = None
EXCLUDE_GSM_IDS = []
MIN_GROUP_N = 3

# 任意の共変量解析（通常はコメントアウト）
COVARIATES = []
# COVARIATES = ["age"]
# COVARIATES = ["sex"]
# COVARIATES = ["age", "sex"]
COVARIATE_OVERRIDE_CSV = None      # 任意CSV: gsm_id,age,sex
COVARIATE_MISSING = "error"         # error / drop

DATA_DIR = "data"
RESULTS_DIR = "results"

# マイクロアレイ
EXPRESSION_SCALE = "unknown"       # unknown / log2 / linear（元データを確認した場合のみ変更）
GENE_COLUMN_OVERRIDES = {}

# RNA-seq
RNA_SEQ_SOURCE = "ncbi_raw"        # ncbi_raw / ncbi_tpm / local_raw / local_tpm / local_log2
RNA_SEQ_MATRIX_PATH = None
RNA_SEQ_FEATURE_COLUMN = None
RNA_SEQ_GENE_MAP_CSV = None         # feature_id,gene_id
RNA_SEQ_SAMPLE_MAP_CSV = None       # matrix_column,gsm_id
RNA_SEQ_SEPARATOR = "auto"
RNA_SEQ_PSEUDOCOUNT = 1.0
RNA_SEQ_MAX_DOWNLOAD_MB = 1500
SHOW_PLOTS = True



In [ ]:
from pathlib import Path
import re, json, warnings
from urllib.parse import urlencode
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import GEOparse
from scipy.stats import ttest_ind, t as student_t
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
from IPython.display import display

assert re.fullmatch(r"GSE\d+", str(GSE_ID), re.I), "GSE_IDはGSE12345形式で指定してください"
assert str(GENE_ID).isdigit(), "GENE_IDはNCBI Gene ID（整数）を指定してください"
assert TECHNOLOGY in {"auto", "microarray", "rna_seq"}
assert EXPRESSION_SCALE in {"unknown", "log2", "linear"}
assert RNA_SEQ_SOURCE in {"ncbi_tpm", "ncbi_raw", "local_tpm", "local_raw", "local_log2"}
assert RNA_SEQ_PSEUDOCOUNT > 0
assert MIN_GROUP_N >= 2
assert isinstance(COVARIATES, list) and len(set(COVARIATES)) == len(COVARIATES)
assert set(COVARIATES).issubset({"age", "sex"}), 'COVARIATESはage、sexだけ指定可能です'
assert COVARIATE_MISSING in {"error", "drop"}
assert GROUP_MODE in {"auto", "csv"}
assert isinstance(POOL_MULTIPLE_DISEASE_GROUPS, bool)
GSE_ID, GENE_ID = str(GSE_ID).upper(), int(GENE_ID)
root = Path(RESULTS_DIR) / f"{GSE_ID}_GeneID_{GENE_ID}_case_control"
root.mkdir(parents=True, exist_ok=True)
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

def norm_key(x):
    return re.sub(r"[^a-z0-9]", "", str(x).lower())

def norm_val(x):
    return re.sub(r"\s+", " ", str(x).strip()).casefold()

def gsm_attributes(gsm):
    """GEOparseのCharacteristics, title, source_name等を安全に整理する。"""
    attrs = {}
    for field, values in gsm.metadata.items():
        if field.startswith("characteristics_ch"):
            for value in values:
                s = str(value)
                if ":" not in s:
                    continue
                k, v = s.split(":", 1)
                k, v = norm_key(k), v.strip()
                if k and v:
                    attrs.setdefault(k, set()).add(v)
        if field in {"title", "source_name_ch1", "organism_ch1", "platform_id", "library_strategy",
                     "age", "age_years", "sex", "gender", "donor_age", "donor_sex"}:
            for v in values:
                if str(v).strip():
                    attrs.setdefault(norm_key(field), set()).add(str(v).strip())
    return {k: sorted(v) for k, v in attrs.items()}

def labels_match(attrs, field, options):
    key = norm_key(field)
    candidates = {norm_val(v) for v in options}
    return any(norm_val(v) in candidates for v in attrs.get(key, []))

def match_filters(attrs, filters):
    return all(labels_match(attrs, k, values) for k, values in filters.items())

def load_overrides(path):
    if not path:
        raise ValueError('GROUP_MODE="csv"ではGROUP_OVERRIDE_CSVを指定してください')
    tab = pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    if not {'gsm_id','group'}.issubset(tab.columns):
        raise ValueError('GROUP_OVERRIDE_CSVにはgsm_id,group列が必要です')
    tab.gsm_id = tab.gsm_id.str.strip().str.upper()
    tab.group = tab.group.str.strip()
    if tab.gsm_id.duplicated().any():
        raise ValueError('GROUP_OVERRIDE_CSVのgsm_idが重複しています')
    if not tab.group.isin(['Normal','Disease','Exclude']).all():
        raise ValueError('group列にはNormal、Disease、Excludeのみ指定してください')
    return dict(zip(tab.gsm_id, tab.group))

# 年齢・性別はGSMの明示的な属性だけを読み取る（タイトルなどから推測しない）。
AGE_KEYS = {norm_key(k) for k in ["age", "age years", "age (years)", "age (yrs)",
                                  "donor age", "patient age", "subject age", "age at biopsy"]}
SEX_KEYS = {norm_key(k) for k in ["sex", "gender", "biological sex", "donor sex",
                                  "patient sex", "subject sex"]}

def parse_age_years(value):
    s = norm_val(value).replace(',', '.').strip()
    if not s or any(x in s for x in ['unknown', 'not available', 'n/a', 'not reported']):
        return None
    # 例: 42, 42.5, 42 years, 2yr3mos, 6ys11mos, 18 months
    simple = re.fullmatch(r'(\d+(?:\.\d+)?)\s*(?:years?|yrs?|ys?|y)?(?:\s*old)?', s)
    if simple:
        n = float(simple.group(1))
        return n if 0 <= n <= 130 else None
    combined = re.fullmatch(
        r'(\d+(?:\.\d+)?)\s*(?:years?|yrs?|ys?|y)'
        r'(?:\s*(?:,|and)?\s*(\d+(?:\.\d+)?)\s*(?:months?|mos?|mths?|m))?'
        r'(?:\s*old)?', s)
    if combined:
        years = float(combined.group(1)); months = float(combined.group(2) or 0)
        n = years + months / 12
        return n if 0 <= n <= 130 and 0 <= months < 12 else None
    months_only = re.fullmatch(r'(\d+(?:\.\d+)?)\s*(?:months?|mos?|mths?)\s*(?:old)?', s)
    if months_only:
        n = float(months_only.group(1)) / 12
        return n if 0 <= n <= 130 else None
    return None

def parse_sex(value):
    s = norm_val(value)
    if s in {'female', 'f'}: return 'Female'
    if s in {'male', 'm'}: return 'Male'
    return None

def extract_covariate(attrs, kind):
    keys = AGE_KEYS if kind == 'age' else SEX_KEYS
    parse = parse_age_years if kind == 'age' else parse_sex
    matches = []
    for k in keys:
        for raw in attrs.get(k, []):
            val = parse(raw)
            if val is not None:
                matches.append((val, k))
    values = set(v for v, _ in matches)
    if len(values) == 1:
        val = next(iter(values))
        return val, 'GSM:' + '|'.join(sorted({k for _, k in matches}))
    if len(values) > 1:
        return np.nan if kind == 'age' else None, 'CONFLICTING_VALUES'
    return np.nan if kind == 'age' else None, 'MISSING_OR_UNRECOGNIZED'

def apply_covariate_overrides(frame, file_path):
    if not file_path:
        return frame
    override = pd.read_csv(file_path, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    if 'gsm_id' not in override.columns or not ({'age', 'sex'} & set(override.columns)):
        raise ValueError('COVARIATE_OVERRIDE_CSVにはgsm_idとage/sexの少なくとも1列が必要です')
    override['gsm_id'] = override['gsm_id'].str.strip().str.upper()
    if override.gsm_id.duplicated().any():
        raise ValueError('COVARIATE_OVERRIDE_CSVのgsm_idが重複しています')
    unknown = set(override.gsm_id) - set(frame.gsm_id)
    if unknown:
        raise ValueError(f'補完CSVに選択群以外のGSMがあります: {sorted(unknown)[:5]}')
    frame = frame.copy().set_index('gsm_id')
    for row in override.to_dict('records'):
        gid = row['gsm_id']
        for key in COVARIATES:
            raw = str(row.get(key, '')).strip()
            if not raw: continue
            val = parse_age_years(raw) if key == 'age' else parse_sex(raw)
            if val is None:
                raise ValueError(f'補完CSVの{gid}、{key}の値を解釈できません: {raw}')
            frame.at[gid, key] = val
            frame.at[gid, key + '_source'] = 'override_csv'
    return frame.reset_index()

def validate_covariates(frame):
    frame = apply_covariate_overrides(frame, COVARIATE_OVERRIDE_CSV)
    missing = pd.Series(False, index=frame.index)
    for key in COVARIATES:
        missing |= frame[key].isna()
    frame['covariate_status'] = np.where(missing, 'MISSING_REQUIRED', 'OK')
    frame.to_csv(root/'covariate_audit.csv', index=False, encoding='utf-8-sig')
    if missing.any():
        detail = frame.loc[missing, ['gsm_id','group','age','age_source','sex','sex_source']]
        display(detail.head(20))
        if COVARIATE_MISSING == 'error':
            raise ValueError(f'選択した共変量{COVARIATES}に欠損があります。'
                             f'{root/"covariate_audit.csv"}を確認し、補完CSVを指定するか'
                             'COVARIATE_MISSING="drop"を選択してください。')
        warnings.warn(f'必要な共変量が欠損した{int(missing.sum())}サンプルを除外しました。')
        frame = frame.loc[~missing].copy()
    if COVARIATES:
        counts_after = frame.group.value_counts().reindex(['Normal','Disease'], fill_value=0)
        if (counts_after < MIN_GROUP_N).any():
            raise ValueError(f'共変量欠損の処理後に群サンプルが不足しました: {counts_after.to_dict()}')
    return frame

# 疾患の固有名は使用せず、GEO内の「疾患・診断・状態」属性を探索する。
# 健常と明示されている値だけNormal、対照と異なる単一ラベルをDiseaseとする。
# 不確実・混在の場合は解析前に停止し、監査CSV/手動2群指定を求める。
GROUP_FIELD_PRIORITY = [
    'diseasestate', 'diseasestatus', 'diseasegroup', 'diseasecondition',
    'diagnosis', 'clinicaldiagnosis', 'diagnosticgroup', 'diagnosticstatus',
    'patientcondition', 'patientscondition', 'patientstatus', 'healthstatus',
    'clinicalstatus', 'clinicalgroup', 'disease', 'healthcondition',
    'casecontrolstatus', 'casecontrol', 'phenotype', 'condition', 'group',
    'cohort', 'status', 'samplegroup',
]
NORMAL_VALUES = {
    'normal', 'normal control', 'normal controls', 'normal group', 'normal subject',
    'healthy', 'healthy control', 'healthy controls', 'healthy donor',
    'healthy donors', 'healthy volunteer', 'healthy volunteers',
    'control', 'controls', 'control group', 'control subject', 'ctrl', 'hc',
    'unaffected', 'unaffected control', 'non diseased', 'non diseased control',
    'nondiseased', 'no disease', 'disease free', 'non dementia',
    'nondemented', 'non demented', 'no diagnosis', 'without disease',
    'healthy individual', 'healthy subjects', 'normal tissue',
}
UNCLASSIFIABLE_VALUES = {
    '', 'nan', 'na', 'n a', 'none', 'null', 'unknown', 'unk', 'not available',
    'not applicable', 'not reported', 'not determined', 'unassigned',
    'indeterminate', 'uncertain', 'other', 'mixed', 'missing',
    'pending', 'intermediate', 'borderline',
    'treated', 'untreated', 'treatment', 'vehicle', 'placebo',
    'mock', 'stimulated', 'unstimulated', 'pre treatment', 'post treatment',
    'lesional', 'non lesional', 'before treatment', 'after treatment',
}

def norm_group_value(v):
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9]+',' ', str(v).casefold())).strip()

def is_normal_value(v):
    return norm_group_value(v) in NORMAL_VALUES

def classify_group_value(v):
    key = norm_group_value(v)
    if key in NORMAL_VALUES:
        return 'Normal'
    if key in UNCLASSIFIABLE_VALUES:
        return 'Exclude'
    return 'Disease' # 疾患名は問わず、診断属性におけるNormal以外の単一カテゴリ

def collect_samples(gse):
    info = {}
    for gsm_id, gsm in sorted(gse.gsms.items()):
        attrs = gsm_attributes(gsm)
        platform = gsm.metadata.get('platform_id', [''])[0] if gsm.metadata.get('platform_id') else ''
        age, age_source = extract_covariate(attrs, 'age')
        sex, sex_source = extract_covariate(attrs, 'sex')
        info[gsm_id] = dict(gsm_id=gsm_id,platform_id=platform,
                            age=age,age_source=age_source,sex=sex,sex_source=sex_source,
                            title=' | '.join(map(str,gsm.metadata.get('title',[]))),
                            attributes=attrs)
    return info

def auto_select_field(infos):
    fields = {f for z in infos.values() for f in z['attributes']}
    if GROUP_FIELD_OVERRIDE:
        f = norm_key(GROUP_FIELD_OVERRIDE)
        if f not in fields:
            raise ValueError('GROUP_FIELD_OVERRIDEがGEOに存在しません。'
                             ' raw_metadata_audit.csvのattr_列を確認してください')
        candidates = [f]
    else:
        candidates = [f for f in GROUP_FIELD_PRIORITY if f in fields]
    reports = []
    for field in candidates:
        groups = {}
        for z in infos.values():
            values = [norm_group_value(v) for v in z['attributes'].get(field, [])]
            values = list(dict.fromkeys(v for v in values if v and v not in UNCLASSIFIABLE_VALUES))
            for v in values:
                groups[v] = groups.get(v, 0) + 1
        normal_labels = {v for v in groups if is_normal_value(v)}
        disease_labels = {v for v in groups if v not in normal_labels}
        n_normal = sum(groups[v] for v in normal_labels)
        n_disease = sum(groups[v] for v in disease_labels)
        if not normal_labels or not disease_labels:
            status = 'ONE_GROUP_OR_MISSING'
        elif len(disease_labels) > 1 and not POOL_MULTIPLE_DISEASE_GROUPS:
            status = 'MULTIPLE_DISEASE_CATEGORIES'
        else:
            status = 'OK'
        reports.append(dict(field=field,status=status,
                            n_normal_labels=len(normal_labels),n_disease_labels=len(disease_labels),
                            normal_samples=n_normal,disease_samples=n_disease,
                            normal_values=' | '.join(sorted(normal_labels)),
                            disease_values=' | '.join(sorted(disease_labels)),
                            priority=GROUP_FIELD_PRIORITY.index(field) if field in GROUP_FIELD_PRIORITY else -1))
    report = pd.DataFrame(reports,columns=['field','status','n_normal_labels','n_disease_labels',
                            'normal_samples','disease_samples','normal_values','disease_values','priority'])
    report.to_csv(root/'group_field_audit.csv',index=False,encoding='utf-8-sig')
    print('候補属性の判定（群名は最終的にNormal/Diseaseへ統一）:')
    display(report.head(25))
    if report.empty:
        raise ValueError('診断・疾患状態を表す属性が見つかりません。'
                         'raw_metadata_audit.csvを確認してGROUP_FIELD_OVERRIDEを指定するか、'
                         'GROUP_MODE="csv"でgsm_id,groupを指定してください。')
    # より優先度の高い診断属性が候補として成立していれば、低優先度へ逃げて誤判定しない。
    viable = report.loc[(report.normal_samples > 0) & (report.disease_samples > 0)]
    if viable.empty:
        raise ValueError('NormalとDiseaseの両群を確認できる属性がありません。'
                         'group_field_audit.csvを確認し、必要ならGROUP_MODE="csv"にしてください。')
    best = viable.iloc[0]
    if best.status != 'OK':
        raise ValueError(f'{best.field}には複数の疾患状態カテゴリがあります。'
                         '無条件の混合は避けています。'
                         '同一Disease群として扱うことが研究上妥当なら'
                         'POOL_MULTIPLE_DISEASE_GROUPS=True、'
                         '個別に選ぶならGROUP_MODE="csv"を使ってください。')
    return str(best.field)

def create_audit_table(infos):
    records = []
    for x in infos.values():
        r = {k:v for k,v in x.items() if k!='attributes'}
        attrs=x['attributes']
        r['attributes_json']=json.dumps(attrs,ensure_ascii=False)
        r.update({'attr_'+k: ' | '.join(vals) for k,vals in attrs.items()})
        records.append(r)
    return pd.DataFrame(records)

def select_groups(infos, field=None, manual=None):
    rows=[]
    for z in infos.values():
        attrs=z['attributes']
        raw_values=attrs.get(field,[]) if field else []
        labels=[classify_group_value(v) for v in raw_values] if field else []
        if manual is not None:
            group=manual.get(z['gsm_id'],'Exclude')
            reason='manual_csv' if z['gsm_id'] in manual else 'not_in_manual_csv'
        elif len(set(labels))==1 and labels[0] != 'Exclude':
            group=labels[0];reason='auto_classified'
        else:
            group='Exclude';reason='missing_or_conflicting_label'
        if INCLUDE_GSM_IDS is not None and z['gsm_id'] not in set(INCLUDE_GSM_IDS):
            group='Exclude';reason='not_in_include_list'
        if z['gsm_id'] in set(EXCLUDE_GSM_IDS):
            group='Exclude';reason='explicit_exclude'
        if group in {'Normal','Disease'} and not match_filters(attrs, COMMON_FILTERS):
            group='Exclude';reason='common_filter'
        if group in {'Normal','Disease'} and not match_filters(attrs, GROUP_FILTERS.get(group, {})):
            group='Exclude';reason='group_filter'
        row={k:v for k,v in z.items() if k!='attributes'}
        row.update(group=group,selection_reason=reason,
                   classification_field=field or 'manual_csv',
                   classification_value=' | '.join(raw_values),
                   attributes_json=json.dumps(attrs,ensure_ascii=False))
        row.update({'attr_'+k:' | '.join(vals) for k, vals in attrs.items()})
        rows.append(row)
    return pd.DataFrame(rows)

gse = GEOparse.get_GEO(geo=GSE_ID, destdir=DATA_DIR, silent=True)
if not gse.gsms:
    raise ValueError('GSEにGSMサンプルがありません')
all_samples = collect_samples(gse)
raw_audit = create_audit_table(all_samples)
raw_audit.to_csv(root/'raw_metadata_audit.csv',index=False,encoding='utf-8-sig')
selected_field = None
if GROUP_MODE == 'auto':
    selected_field = auto_select_field(all_samples)
    metadata_all = select_groups(all_samples,selected_field)
else:
    manual = load_overrides(GROUP_OVERRIDE_CSV)
    unknown = set(manual) - set(all_samples)
    if unknown:
        raise ValueError(f'手動CSVにこのGSEには存在しないGSMがあります: {sorted(unknown)[:10]}')
    metadata_all = select_groups(all_samples,manual=manual)
metadata_all.to_csv(root/'sample_metadata_audit.csv',index=False,encoding='utf-8-sig')
print('Auto field:', selected_field, '| Groups:')
display(metadata_all.groupby(['group','selection_reason']).size().rename('n').reset_index())
metadata = metadata_all.loc[metadata_all.group.isin(['Normal','Disease'])].copy()
counts = metadata.group.value_counts().reindex(['Normal','Disease'],fill_value=0)
if (counts < MIN_GROUP_N).any():
    print('診断:', metadata_all.selection_reason.value_counts().to_dict())
    print('分類値:',metadata_all.classification_value.value_counts().head(20).to_dict())
    raise ValueError(f'抽出サンプル不足: {counts.to_dict()}。'
                     f'{root/"sample_metadata_audit.csv"}とgroup_field_audit.csvを確認してください')
if COVARIATES:
    metadata = validate_covariates(metadata)
    print('共変量:', COVARIATES, '使用GSM:', len(metadata))



In [ ]:
def classify_assay(metadata_dict):
    val = " ".join(str(v) for key in ("type", "library_strategy", "library_selection", "technology")
                   for v in metadata_dict.get(key, [])).lower()
    if any(x in val for x in ("rna-seq", "rna seq", "expression profiling by high throughput sequencing", "transcriptomic")):
        return "rna_seq"
    if any(x in val for x in ("expression profiling by array", "microarray")):
        return "microarray"
    return "unknown"

metadata["assay_detected"] = [classify_assay(gse.gsms[gsm_id].metadata)
                               for gsm_id in metadata.gsm_id]
mask = metadata.assay_detected == "unknown"
metadata.loc[mask, "assay_detected"] = classify_assay(gse.metadata)
metadata.to_csv(root / "metadata_selected.csv", index=False, encoding="utf-8-sig")
known = set(metadata.assay_detected) - {"unknown"}
if TECHNOLOGY == "auto":
    if len(known) != 1 or "unknown" in set(metadata.assay_detected):
        raise ValueError("実験技術を一意に判定できません。metadata_selected.csvを確認し対象GSMを絞ってください。")
    assay = known.pop()
else:
    assay = TECHNOLOGY
    conflict = metadata.assay_detected.isin(["rna_seq", "microarray"]) & (metadata.assay_detected != assay)
    if conflict.any():
        raise ValueError("TECHNOLOGY指定とGSMの技術が異なります。対象GSMを選び直してください。")
print("Selected:", counts.to_dict(), "Technology:", assay)



In [ ]:
# GPLの候補列を複数探索し、Gene IDの完全一致で対応プローブを特定する。
# 注: 遺伝子シンボルやRefSeq accessionの数値をGene IDと混同しない。
GENE_ID_ALIASES = ['gene_assignment','entrez_gene_id','entrez gene id','entrez_gene',
    'gene id','gene_id','geneid','ncbi_gene_id','ncbi gene id','entrez id',
    'entrezgene','entrezgeneid','geneid (entrez)','entrez_gene_ids']

def normalize_col(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

def candidate_gene_columns(gpl_id, ann):
    if gpl_id in GENE_COLUMN_OVERRIDES:
        override = GENE_COLUMN_OVERRIDES[gpl_id]
        requested = [override] if isinstance(override, str) else list(override)
        missing = [c for c in requested if c not in ann.columns]
        if missing:
            raise ValueError(f'{gpl_id}: 指定注釈列が存在しません: {missing}')
        return requested
    alias_keys = {normalize_col(a) for a in GENE_ID_ALIASES}
    return [c for c in ann.columns if normalize_col(c) in alias_keys]

def match_gene(v, gene_id, col):
    if pd.isna(v):
        return False
    value = str(v).strip()
    target = str(int(gene_id))
    if normalize_col(col) == 'geneassignment':
        for record in re.split(r'\s*///\s*', value):
            parts = [p.strip() for p in record.split('//')]
            # GPL5175等のgene_assignment: 5番目の要素がEntrez Gene ID
            if len(parts) >= 5 and target in re.findall(r'(?<!\d)\d+(?!\d)', parts[4]):
                return True
        return False
    # 非数値の複合注釈からIDを誤認しないよう、区切り文字に分離された整数だけ許容
    tokens = re.split(r'\s*(?:///|[;,|])\s*', value)
    return target in [t.strip() for t in tokens]



In [ ]:
def build_microarray_mapping():
    mapping = []
    annotation_report = []
    for gpl_id in metadata.platform_id.dropna().unique():
        gpl = gse.gpls.get(gpl_id)
        if gpl is None:
            annotation_report.append(dict(gpl_id=gpl_id,status='GPL_NOT_LOADED',column='',n=0,
                                          available_columns='',comment='GPLオブジェクトが見つかりません'))
            continue
        a = gpl.table
        columns = candidate_gene_columns(gpl_id, a)
        if 'ID' not in a.columns or not columns:
            annotation_report.append(dict(gpl_id=gpl_id,status='ANNOTATION_COLUMN_NOT_FOUND',column='',n=0,
                                          available_columns=' | '.join(map(str,a.columns)),
                                          comment='GPLの注釈列を確認してGENE_COLUMN_OVERRIDESを設定'))
            print(f'{gpl_id} 注釈列候補:',list(a.columns))
            continue
        matches_by_col = {}
        for col in columns:
            mask = a[col].map(lambda v: match_gene(v, GENE_ID, col))
            ids = set(a.loc[mask, 'ID'].dropna().astype(str))
            matches_by_col[col] = ids
            annotation_report.append(dict(gpl_id=gpl_id,
                                          status='OK' if ids else 'GENE_NOT_FOUND',
                                          column=col,n=len(ids),
                                          available_columns=' | '.join(map(str,a.columns)),
                                          comment='列別の照合結果'))
        # 異なる注釈列間の不整合があれば監査対象として警告
        nonempty = [s for s in matches_by_col.values() if s]
        if len(nonempty)>1 and any(s != nonempty[0] for s in nonempty[1:]):
            warnings.warn(f'{gpl_id}: Gene ID対応プローブが注釈列間で一致しません。annotation_audit.csvを確認してください')
        union = set().union(*matches_by_col.values())
        for probe_id in sorted(union):
            sources = [c for c, ids in matches_by_col.items() if probe_id in ids]
            mapping.append(dict(platform_id=gpl_id, probe_id=probe_id,
                                annotation_column=' | '.join(sources)))
    annotation_report = pd.DataFrame(annotation_report)
    annotation_report.to_csv(root/'annotation_audit.csv',index=False,encoding='utf-8-sig')
    print('注釈の検査:')
    display(annotation_report[['gpl_id','status','column','n','comment']])
    if not mapping:
        raise ValueError(
            f'Gene ID {GENE_ID} に対応するプローブが見つかりません。'
            f' {root / "annotation_audit.csv"} のstatusとavailable_columnsを確認してください。'
            ' GPL側にGene ID注釈がない場合は、公式のGPL注釈ファイルなどで対応表を確認する必要があります。'
        )
    mapping = pd.DataFrame(mapping)
    mapping.to_csv(root/'probe_mapping.csv',index=False,encoding='utf-8-sig')
    display(mapping.head(20))
    return mapping



In [ ]:
def geo_rnaseq_filename(gse_id, source):
    suffix = {'ncbi_tpm': 'norm_counts_TPM_GRCh38.p13_NCBI.tsv.gz',
              'ncbi_raw': 'raw_counts_GRCh38.p13_NCBI.tsv.gz'}[source]
    return f'{gse_id}_{suffix}'

def download_ncbi_rnaseq(gse_id, source):
    organisms = set()
    for gsm_id in metadata.gsm_id:
        organisms.update(map(str, gse.gsms[gsm_id].metadata.get('organism_ch1', [])))
    if organisms != {'Homo sapiens'}:
        raise ValueError('NCBI自動取得は本NotebookではHomo sapiens専用です。organism_ch1の確認後、ローカル行列を指定してください。')
    fname = geo_rnaseq_filename(gse_id, source)
    folder = Path(DATA_DIR) / gse_id
    folder.mkdir(parents=True,exist_ok=True)
    file = folder / fname
    if file.exists():
        with file.open('rb') as f: magic=f.read(2)
        if magic != b'\x1f\x8b':
            raise ValueError(f'キャッシュファイルがgzipではありません: {file}。削除して再実行してください。')
        return file, 'cache'
    params = {'type': 'rnaseq_counts','acc':gse_id,'format':'file','file':fname}
    url = 'https://www.ncbi.nlm.nih.gov/geo/download/?' + urlencode(params)
    part = file.with_name(file.name+'.part')
    limit = int(RNA_SEQ_MAX_DOWNLOAD_MB*1024*1024)
    total=0
    try:
        with requests.get(url,stream=True,timeout=(20,120),headers={'User-Agent':'GeoCaseControlNotebook/1.0'}) as r:
            r.raise_for_status()
            ct = r.headers.get('content-type','').lower()
            with part.open('wb') as w:
                for buf in r.iter_content(chunk_size=1024*1024):
                    if not buf:continue
                    total+=len(buf)
                    if total>limit:raise ValueError(f'ファイルが上限{RNA_SEQ_MAX_DOWNLOAD_MB}MBを超過')
                    w.write(buf)
        with part.open('rb') as f: magic=f.read(2)
        if magic != b'\x1f\x8b':
            raise ValueError('GEOからgzip形式のRNA-seq行列を取得できませんでした（未生成・URL変更等）。'+
                             f'Content-Type={ct}。GEOのDownload Dataを確認するかローカル行列を指定してください。')
        part.replace(file)
    finally:
        part.unlink(missing_ok=True)
    return file,url

def read_sample_mapping(file_path):
    if not file_path:return {}
    tab=pd.read_csv(file_path, dtype=str)
    if not {'matrix_column','gsm_id'}.issubset(tab.columns):
        raise ValueError('RNA_SEQ_SAMPLE_MAP_CSVにはmatrix_column,gsm_idが必要')
    if tab.matrix_column.duplicated().any() or tab.gsm_id.duplicated().any():
        raise ValueError('サンプル対応表に重複があります。1試料1列にしてください')
    return dict(zip(tab.matrix_column,tab.gsm_id))

def read_gene_mapping(file_path,target_gene_id):
    if not file_path:return None
    tab=pd.read_csv(file_path,dtype=str)
    if not {'feature_id','gene_id'}.issubset(tab.columns):
        raise ValueError('RNA_SEQ_GENE_MAP_CSVにはfeature_id,gene_idが必要')
    tab['gene_id']=tab['gene_id'].str.strip()
    features=tab.loc[tab.gene_id == str(target_gene_id),'feature_id'].astype(str).str.strip()
    if not len(features):
        raise ValueError(f'遺伝子対応表にはGene ID {target_gene_id}がありません')
    return set(features)

def detect_delimiter(path):
    if RNA_SEQ_SEPARATOR != 'auto':return RNA_SEQ_SEPARATOR
    name=path.name.lower()
    return ',' if (name.endswith('.csv') or name.endswith('.csv.gz')) else '\t'

def match_column_to_gsm(name, explicit):
    label=str(name)
    if label in explicit:return explicit[label]
    if re.fullmatch(r'GSM\d+',label,re.I):return label.upper()
    ids=set(re.findall(r'GSM\d+',label,re.I))
    return next(iter(ids)).upper() if len(ids)==1 else None

def extract_rnaseq_target(matrix_path, gene_id, mode, selected_gsms):
    """Read a single target feature but normalize raw counts using all feature rows."""
    matrix_path=Path(matrix_path)
    if not matrix_path.is_file():raise FileNotFoundError(matrix_path)
    sep=detect_delimiter(matrix_path)
    # First line only, preserve feature ID as literal text including leading zeros.
    header=pd.read_csv(matrix_path,sep=sep,nrows=0,encoding='utf-8-sig')
    names=header.columns.tolist()
    if len(names)<2:raise ValueError('RNA-seq行列にはfeature ID列と試料列が必要です')
    feature_col=RNA_SEQ_FEATURE_COLUMN or names[0]
    if feature_col not in names:raise ValueError(f'feature列がありません: {feature_col}。実際の列={names[:20]}')
    explicit=read_sample_mapping(RNA_SEQ_SAMPLE_MAP_CSV)
    sel=set(selected_gsms)
    sample_cols={}
    for n in names:
        if n == feature_col:continue
        gsm_id=match_column_to_gsm(n,explicit)
        if gsm_id in sel:
            if gsm_id in sample_cols:
                raise ValueError(f'{gsm_id}に対応する複数列があります。RNA_SEQ_SAMPLE_MAP_CSVで1対1にしてください')
            sample_cols[gsm_id]=n
    if not sample_cols:
        raise ValueError('行列中にGSM列がありません。列名を確認しRNA_SEQ_SAMPLE_MAP_CSVでmatrix_column,gsm_idを対応させてください。')
    missing=sel-set(sample_cols)
    if missing:warnings.warn(f'RNA-seq行列に存在しないGSMが{len(missing)}件: {sorted(missing)[:10]}（除外します）')
    print('RNA-seq対応GSM:',len(sample_cols),'/選択GSM:',len(selected_gsms))
    targets=read_gene_mapping(RNA_SEQ_GENE_MAP_CSV,gene_id)
    if targets is None:targets={str(gene_id)}
    values_by_gsm = {}
    libsize = pd.Series(0.,index=list(sample_cols))
    n_target_rows=0
    count_rows=0
    for chunk in pd.read_csv(matrix_path,sep=sep,usecols=[feature_col,*sample_cols.values()],
                             dtype={feature_col:str},chunksize=2000,encoding='utf-8-sig'):
        count_rows+=len(chunk)
        vals=chunk[list(sample_cols.values())].apply(pd.to_numeric,errors='coerce')
        bad = ~np.isfinite(vals.to_numpy(dtype=float))
        if mode == 'raw':
            if bad.any() or (vals.to_numpy(dtype=float)<0).any():
                raise ValueError('raw countsに非数値・欠損・負値があります。行列形式を確認してください。')
            libsize+=pd.Series(vals.sum(axis=0).to_numpy(dtype=float),index=list(sample_cols))
        hit=chunk[feature_col].astype(str).str.strip().isin(targets)
        if hit.any():
            n_target_rows+=int(hit.sum())
            for _, row in vals.loc[hit].iterrows():
                for gsm_id, col_name in sample_cols.items():
                    values_by_gsm.setdefault(gsm_id,[]).append(float(row[col_name]))
    if n_target_rows == 0:
        raise ValueError(f'Gene ID {gene_id}（対応features={sorted(targets)[:6]}）がRNA-seq行列のfeature列にありません。'+
                         ' Ensembl等のIDならRNA_SEQ_GENE_MAP_CSVを設定してください。')
    if n_target_rows > 1:
        raise ValueError(f'Gene ID {gene_id} に対応する行が{n_target_rows}件です。'+
                         ' transcript/重複featureの集約規則を人が指定する必要があります。自動で合算しません。')
    if mode == 'raw' and (libsize<=0).any():
        raise ValueError('raw countsのライブラリサイズが0の試料が存在します')
    out=[]
    for gsm_id in selected_gsms:
        if gsm_id not in values_by_gsm:continue
        raw=values_by_gsm[gsm_id][0]
        if not np.isfinite(raw) or raw<0:
            continue
        if mode=='raw':
            value=raw/libsize[gsm_id]*1_000_000
            metric='CPM'
        elif mode=='tpm':
            value=raw
            metric='TPM'
        else:
            value=raw
            metric='log2_preprocessed'
        if mode=='log2':
            transformed=value
        else:
            transformed=float(np.log2(value+RNA_SEQ_PSEUDOCOUNT))
        out.append(dict(gsm_id=gsm_id,probe_id=str(gene_id),feature_id=next(iter(targets)),
                        untransformed_value=float(raw),normalized_value=float(value),
                        expression=float(transformed),library_size=float(libsize[gsm_id]) if mode=='raw' else np.nan))
    return pd.DataFrame(out),{'n_matrix_rows':count_rows,'n_matched_features':n_target_rows,'n_samples_matrix':len(sample_cols),
                              'n_samples_used':len(out),'unmatched_gsms':sorted(missing)}

def get_rnaseq_expression():
    if RNA_SEQ_SOURCE.startswith('ncbi_'):
        path, provenance=download_ncbi_rnaseq(GSE_ID,RNA_SEQ_SOURCE)
        mode='raw' if RNA_SEQ_SOURCE=='ncbi_raw' else 'tpm'
    else:
        if not RNA_SEQ_MATRIX_PATH:raise ValueError('local_*の利用にはRNA_SEQ_MATRIX_PATHが必要です')
        path=Path(RNA_SEQ_MATRIX_PATH)
        provenance=str(path.resolve())
        mode={'local_raw':'raw','local_tpm':'tpm','local_log2':'log2'}[RNA_SEQ_SOURCE]
    df, audit=extract_rnaseq_target(path,GENE_ID,mode,metadata.gsm_id.tolist())
    if df.empty:raise ValueError('選択されたGSMに対象遺伝子のRNA-seq発現値がありません')
    meta=metadata[['gsm_id','platform_id','group','age','sex']]
    df=df.merge(meta,on='gsm_id',validate='one_to_one',how='inner')
    df['gene_id']=GENE_ID
    df['data_type']='rna_seq'
    audit.update({'data_source':RNA_SEQ_SOURCE,'data_path':str(path),'provenance':provenance,
                  'metric':'log2(CPM+pseudocount)' if mode=='raw' else
                           'log2(TPM+pseudocount)' if mode=='tpm' else 'user_supplied_log2'})
    (root/'rna_seq_audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')
    print('RNA-seqデータ:',audit['metric'], '\nソース:',path)
    return df, audit['metric']



In [ ]:
def get_microarray_expression(mapping):
    def extract_expression(gsm,probe_ids):
        table=gsm.table
        if not {'ID_REF','VALUE'}.issubset(table.columns):
            return None,'GSM_TABLE_MISSING_ID_REF_OR_VALUE'
        sub=table.loc[table['ID_REF'].astype(str).isin(probe_ids),['ID_REF','VALUE']].copy()
        sub['ID_REF']=sub['ID_REF'].astype(str)
        if sub.ID_REF.duplicated().any():return None,'DUPLICATE_PROBE_ID'
        sub['VALUE']=pd.to_numeric(sub.VALUE,errors='coerce')
        return sub.set_index('ID_REF')['VALUE'],'OK'

    expression=[]; issues=[]
    for gpl_id, mapping_sub in mapping.groupby('platform_id'):
        ids=set(mapping_sub.probe_id)
        for item in metadata.loc[metadata.platform_id==gpl_id].itertuples(index=False):
            gsm=gse.gsms[item.gsm_id]
            data,status=extract_expression(gsm,ids)
            if status!='OK':
                issues.append(dict(gsm_id=item.gsm_id,platform_id=gpl_id,status=status));continue
            for probe in ids:
                v=data.get(probe,np.nan)
                if pd.notna(v) and np.isfinite(v):
                    expression.append(dict(gsm_id=item.gsm_id,platform_id=gpl_id,probe_id=probe,
                        gene_id=GENE_ID,group=item.group,age=item.age,sex=item.sex,
                        expression=float(v)))
                else:issues.append(dict(gsm_id=item.gsm_id,platform_id=gpl_id,status=f'MISSING_EXPRESSION:{probe}'))
    issues=pd.DataFrame(issues,columns=['gsm_id','platform_id','status'])
    issues.to_csv(root/'expression_issues.csv',index=False,encoding='utf-8-sig')
    if not expression:raise ValueError('解析可能なGSM発現値がありません。supplementaryファイルからの行列作成が必要な可能性があります')
    all_expression=pd.DataFrame(expression)
    all_expression.to_csv(root/'expression.csv',index=False,encoding='utf-8-sig')
    return all_expression

if assay == 'microarray':
    mapping=build_microarray_mapping()
    all_expression=get_microarray_expression(mapping)
    all_expression['data_type']='microarray'
    analysis_scale=EXPRESSION_SCALE
else:
    all_expression,analysis_scale=get_rnaseq_expression()
    all_expression.to_csv(root/'expression.csv',index=False,encoding='utf-8-sig')

print('対象発現値:',len(all_expression),'rows / 有効GSM:',all_expression.gsm_id.nunique())
display(all_expression.head(10))



In [ ]:
def compare_two_groups(sub):
    c = sub.loc[sub.group == "Normal", "expression"].dropna().to_numpy(dtype=float)
    d = sub.loc[sub.group == "Disease", "expression"].dropna().to_numpy(dtype=float)
    item = dict(normal_n=len(c), disease_n=len(d),
                normal_mean=np.mean(c) if len(c) else np.nan,
                disease_mean=np.mean(d) if len(d) else np.nan,
                normal_median=np.median(c) if len(c) else np.nan,
                disease_median=np.median(d) if len(d) else np.nan,
                mean_difference=np.mean(d)-np.mean(c) if len(c) and len(d) else np.nan,
                welch_p=np.nan, welch_t=np.nan, welch_df=np.nan,
                ci_95_lower=np.nan, ci_95_upper=np.nan, hedges_g=np.nan,
                status="INSUFFICIENT_SAMPLES")
    if min(len(c), len(d)) < MIN_GROUP_N:
        return item
    var_c, var_d = np.var(c,ddof=1), np.var(d,ddof=1)
    result = ttest_ind(d,c,equal_var=False)
    se2 = var_d/len(d) + var_c/len(c)
    if se2 > 0:
        df = se2**2 / ((var_d/len(d))**2/(len(d)-1) + (var_c/len(c))**2/(len(c)-1))
        half = float(student_t.ppf(0.975, df) * np.sqrt(se2))
        item.update(welch_p=float(result.pvalue),welch_t=float(result.statistic),welch_df=float(df),
                    ci_95_lower=float(item["mean_difference"]-half),
                    ci_95_upper=float(item["mean_difference"]+half), status="OK")
    else:
        item["status"] = "ZERO_VARIANCE"
    pooled = ((len(c)-1)*var_c+(len(d)-1)*var_d)/(len(c)+len(d)-2)
    if pooled>0:
        cohen_d = (np.mean(d)-np.mean(c))/np.sqrt(pooled)
        item["hedges_g"] = float(cohen_d * (1-3/(4*(len(c)+len(d))-9)))
    if assay == "microarray" and EXPRESSION_SCALE == "log2":
        item["log2_fold_change"] = item["mean_difference"]
        item["fold_change"] = float(2**item["mean_difference"])
    else:
        item["log2_fold_change"] = np.nan
        item["fold_change"] = np.nan
    return item


def adjusted_ols(sub):
    result = dict(adjusted_status='NOT_REQUESTED', adjusted_n=np.nan,
                  adjusted_normal_n=np.nan, adjusted_disease_n=np.nan,
                  adjusted_disease_beta=np.nan, adjusted_disease_p=np.nan,
                  adjusted_disease_ci_95_lower=np.nan, adjusted_disease_ci_95_upper=np.nan,
                  age_beta_per_year=np.nan, age_p=np.nan, age_ci_95_lower=np.nan,
                  age_ci_95_upper=np.nan,
                  male_vs_female_beta=np.nan, male_vs_female_p=np.nan,
                  male_vs_female_ci_95_lower=np.nan, male_vs_female_ci_95_upper=np.nan,
                  adjusted_r_squared=np.nan)
    if not COVARIATES:
        return result
    cols = ['group', 'expression', *COVARIATES]
    data = sub[cols].copy().dropna()
    cnt = data.group.value_counts().reindex(['Normal','Disease'], fill_value=0)
    result.update(adjusted_n=len(data), adjusted_normal_n=int(cnt['Normal']),
                  adjusted_disease_n=int(cnt['Disease']))
    if (cnt < MIN_GROUP_N).any():
        result['adjusted_status'] = 'INSUFFICIENT_SAMPLES'
        return result
    x = pd.DataFrame(index=data.index)
    x['disease'] = data.group.eq('Disease').astype(float)
    if 'age' in COVARIATES:
        x['age'] = data['age'].astype(float)
    if 'sex' in COVARIATES:
        x['male'] = data['sex'].eq('Male').astype(float)
    x = sm.add_constant(x, has_constant='add')
    if np.linalg.matrix_rank(x.to_numpy(dtype=float)) < x.shape[1]:
        result['adjusted_status'] = 'SINGULAR_DESIGN'
        return result
    if len(data) - x.shape[1] < 2:
        result['adjusted_status'] = 'LOW_RESIDUAL_DF'
        return result
    try:
        model = sm.OLS(data.expression.astype(float), x).fit(cov_type='HC3', use_t=True)
        ci = model.conf_int(alpha=0.05)
        terms = {'disease': ('adjusted_disease_beta','adjusted_disease_p',
                             'adjusted_disease_ci_95_lower','adjusted_disease_ci_95_upper'),
                 'age': ('age_beta_per_year','age_p','age_ci_95_lower','age_ci_95_upper'),
                 'male': ('male_vs_female_beta','male_vs_female_p',
                          'male_vs_female_ci_95_lower','male_vs_female_ci_95_upper')}
        for term, names in terms.items():
            if term in model.params.index:
                vals = [model.params[term], model.pvalues[term], ci.loc[term,0], ci.loc[term,1]]
                if not np.isfinite(vals).all():
                    result['adjusted_status'] = 'NONFINITE_ESTIMATES'
                    return result
                result.update(dict(zip(names, map(float, vals))))
        result['adjusted_r_squared'] = float(model.rsquared)
        result['adjusted_status'] = 'OK'
    except (ValueError, ArithmeticError, np.linalg.LinAlgError) as exc:
        result['adjusted_status'] = 'MODEL_ERROR:' + type(exc).__name__
    return result

summary = []


In [ ]:
for (platform_id, probe_id), sub in all_expression.groupby(["platform_id", "probe_id"],dropna=False):
    row = dict(gse_id=GSE_ID, gene_id=GENE_ID,
               platform_id=platform_id, probe_id=probe_id, assay=assay,
               expression_metric=analysis_scale)
    row.update(compare_two_groups(sub))
    row.update(adjusted_ols(sub))
    summary.append(row)
summary = pd.DataFrame(summary)
summary["welch_fdr_bh"] = np.nan
valid = summary.welch_p.notna() & np.isfinite(summary.welch_p)
if valid.any():
    summary.loc[valid, "welch_fdr_bh"] = multipletests(summary.loc[valid,"welch_p"],method="fdr_bh")[1]
summary["significant_fdr_0_05"] = summary.welch_fdr_bh < 0.05
if COVARIATES:
    # 各項目について、プローブを跨いだBH-FDRを別々に計算
    for src, target in [('adjusted_disease_p','adjusted_disease_fdr_bh'),
                        ('age_p','age_fdr_bh'), ('male_vs_female_p','sex_fdr_bh')]:
        summary[target] = np.nan
        ok = summary[src].notna() & np.isfinite(summary[src])
        if ok.any():
            summary.loc[ok,target] = multipletests(summary.loc[ok,src],method='fdr_bh')[1]
summary = summary.sort_values("welch_p",na_position="last")
summary.to_csv(root/"summary.csv",index=False,encoding="utf-8-sig")
print("結果:")
display(summary)
if not valid.any():
    warnings.warn("統計検定が1つも成立しませんでした。各群の対象サンプル数、数値、発現行列を確認してください。")

for (platform_id, probe_id), sub in all_expression.groupby(["platform_id", "probe_id"],dropna=False):
    c = sub.loc[sub.group == "Normal","expression"].dropna().to_numpy(dtype=float)
    d = sub.loc[sub.group == "Disease","expression"].dropna().to_numpy(dtype=float)
    if not len(c) or not len(d):
        continue
    fig, ax = plt.subplots(figsize=(6.4,4.4))
    ax.boxplot([c,d], positions=[1,2], widths=0.42, showfliers=False)
    rng = np.random.default_rng(42)
    ax.scatter(1+rng.uniform(-0.13,0.13,len(c)),c,s=15,alpha=0.65,label=f"Normal (n={len(c)})")
    ax.scatter(2+rng.uniform(-0.13,0.13,len(d)),d,s=15,alpha=0.65,label=f"Disease (n={len(d)})")
    ax.set_xticks([1,2],["Normal","Disease"])
    ax.set_ylabel(str(analysis_scale))
    ax.set_title(f"{GSE_ID} | Gene ID {GENE_ID} | {probe_id}")
    ax.legend(loc="best")
    fig.tight_layout()
    filename=f"{re.sub(r'[^A-Za-z0-9_.-]+','_',str(platform_id))}_{re.sub(r'[^A-Za-z0-9_.-]+','_',str(probe_id))}_groups.png"
    fig.savefig(root/filename,dpi=160)
    if SHOW_PLOTS: plt.show()
    plt.close(fig)

    if 'age' in COVARIATES:
        fig, ax = plt.subplots(figsize=(6.4,4.4))
        for group in ['Normal', 'Disease']:
            pts = sub.loc[sub.group.eq(group) & sub.age.notna()]
            ax.scatter(pts.age, pts.expression, label=f'{group} (n={len(pts)})', alpha=0.7)
        ax.set_xlabel('Age (years)')
        ax.set_ylabel(str(analysis_scale))
        ax.set_title(f'{GSE_ID} | Gene ID {GENE_ID} | {probe_id}')
        ax.legend()
        fig.tight_layout()
        filename=f"{re.sub(r'[^A-Za-z0-9_.-]+','_',str(platform_id))}_{re.sub(r'[^A-Za-z0-9_.-]+','_',str(probe_id))}_age.png"
        fig.savefig(root/filename, dpi=160)
        if SHOW_PLOTS: plt.show()
        plt.close(fig)

config = {
    "gse_id":GSE_ID,"gene_id":GENE_ID,"assay":assay,"group_mode":GROUP_MODE,"group_field":selected_field,
    "pooled_disease_categories":POOL_MULTIPLE_DISEASE_GROUPS,
    "common_filters":COMMON_FILTERS,"group_filters":GROUP_FILTERS,
    "override_csv":GROUP_OVERRIDE_CSV,"rna_seq_source":RNA_SEQ_SOURCE if assay=="rna_seq" else None,
    "expression_scale":analysis_scale,"selected_count_by_group":metadata.group.value_counts().to_dict(),
    "covariates":COVARIATES, "covariate_override_csv":COVARIATE_OVERRIDE_CSV,
    "covariate_missing":COVARIATE_MISSING,
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "note":"Exploratory independent-group comparison. Optional age/sex-adjusted OLS uses HC3 robust SE. Not a substitute for count-based DE models or matched-pair analysis."
}
(root/"analysis_settings.json").write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding="utf-8")
print("保存先:",root.resolve())
